In [1]:
import os
HF_TOKEN = os.getenv("HF_TOKEN")
HF_TOKEN[:3]+'...'

'hf_...'

# Load Model

In [5]:
import transformers
import torch

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

bnb_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = transformers.AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
    token=HF_TOKEN,
)

tokenizer = transformers.AutoTokenizer.from_pretrained(
    model_name, 
    token=HF_TOKEN, 
)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [6]:
generator = transformers.pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    pad_token_id=tokenizer.eos_token_id,
    max_new_tokens=1024,
    # do_sample=True,
    # temperature=0.6,
    # top_p=0.9,
)

def get_response(chats): 
    gen_text = generator(chats)[0]  # First return sequence
    return gen_text['generated_text'][-1]['content']

Device set to use cuda:0


In [7]:
messages = [
    {"role": "system", "content": "You are a chatbot to assist user for their tasks."},
    {"role": "user", "content": "What is 1 + 2?"},
]
response = get_response(messages)
print(response)

The answer to 1 + 2 is 3!


In [8]:
import json
import random

def load_jsonlines(file_name: str):
    with open(file_name, 'r') as f:
        return [json.loads(line) for line in f]


def simple_chats(question: str) -> dict:
    def question_prompt(s):
        return f'Question: {s}'

    def answer_prompt(s):
        return f'Answer: {s}'

    chats = [
         {
             "role": "system",
             "content": "You are a grade school math problem solver. At the end, you MUST write the answer as an integer after '####'.",
             #"content": "You are a grade school math problem solver. At the end, you MUST write the answer as an integer after '####'. Let's think step by step.",
         },
    ]

    chats.append({"role": "user", "content": question_prompt(question)})

    return chats


def extract_ans_from_response(answer: str, eos=None):
    if eos:
        answer = answer.split(eos)[0].strip()

    answer = answer.split('####')[-1].strip()

    for remove_char in [',', '$', '%', 'g']:
        answer = answer.replace(remove_char, '')

    try:
        return int(answer)
    except ValueError:
        return answer



train_data = load_jsonlines('data/train.jsonl')
test_data = load_jsonlines('data/test.jsonl')




In [9]:
messages = simple_chats(question=test_data[3]['question'])
print(messages)

response = get_response(messages)
print(response)

pred_ans = extract_ans_from_response(response)
pred_ans

[{'role': 'system', 'content': "You are a grade school math problem solver. At the end, you MUST write the answer as an integer after '####'."}, {'role': 'user', 'content': 'Question: James decides to run 3 sprints 3 times a week.  He runs 60 meters each sprint.  How many total meters does he run a week?'}]
Let's break this down step by step!

James runs 3 sprints 3 times a week, so he runs a total of:

3 sprints/week × 3 times/week = 9 sprints/week

Each sprint is 60 meters, so the total distance he runs in a week is:

9 sprints/week × 60 meters/sprint = 540 meters/week

#### 540


540

In [10]:
# Overall test
from tqdm import tqdm
import os

if not os.path.exists('log'):
    os.makedirs('log')

log_file_path = 'log/errors.txt'
with open(log_file_path, 'w') as log_file:
    log_file.write('')


total = correct = 0
for qna in tqdm(test_data):

    messages = simple_chats(question=qna['question'])
    response = get_response(messages)
    
    pred_ans = extract_ans_from_response(response)
    true_ans = extract_ans_from_response(qna['answer'])
    

    total += 1
    if pred_ans != true_ans:
        with open(log_file_path, 'a', encoding='utf-8') as log_file:
            log_file.write(f"{messages}\n\n")
            log_file.write(f"Response: {response}\n\n")
            log_file.write(f"Ground Truth: {qna['answer']}\n\n")
            log_file.write(f"Current Accuracy: {correct/total:.3f}\n\n")
            log_file.write('\n\n')
    else:
        correct += 1
    print(f"Total: {total}, Correct: {correct}")
    

  0%|          | 1/1319 [00:02<46:21,  2.11s/it]

Total: 1, Correct: 1


  0%|          | 2/1319 [00:03<42:13,  1.92s/it]

Total: 2, Correct: 2


  0%|          | 3/1319 [00:07<1:02:17,  2.84s/it]

...

Total: 1317, Correct: 1003


100%|█████████▉| 1318/1319 [1:09:09<00:02,  2.73s/it]

Total: 1318, Correct: 1004


100%|██████████| 1319/1319 [1:09:12<00:00,  3.15s/it]

Total: 1319, Correct: 1004
